In [2]:
# Cell 1: Install packages (uncomment if needed)
# !pip install pandas numpy scikit-learn mlflow ydata-profiling pandera dvc

import pandas as pd
import numpy as np

# Load Data
df = pd.read_excel('Salary Data (1).xlsx')
print(df.head())

    Age  Gender Education Level          Job Title  Years of Experience  \
0  32.0    Male      Bachelor's  Software Engineer                  5.0   
1  28.0  Female        Master's       Data Analyst                  3.0   
2  45.0    Male             PhD     Senior Manager                 15.0   
3  36.0  Female      Bachelor's    Sales Associate                  7.0   
4  52.0    Male        Master's           Director                 20.0   

     Salary  
0   90000.0  
1   65000.0  
2  150000.0  
3   60000.0  
4  200000.0  


In [6]:
# Cell 2: Data Profiling
from ydata_profiling import ProfileReport
from IPython.display import IFrame

profile = ProfileReport(df, title="Salary Data Profiling Report")
profile.to_file("salary_data_profiling.html")

# Render the HTML file inside the notebook cell
IFrame(src="salary_data_profiling.html", width="100%", height="600")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 6/6 [00:00<00:00, 374.76it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
import pandas as pd
import pandera as pa
from pandera import Column, Check, DataFrameSchema
from sklearn.model_selection import train_test_split

# 1. Clean Duplicates & Missing Values
df_clean = df.drop_duplicates().dropna().copy()

# 2. Validate Cleaned Data Schema
schema = DataFrameSchema({
    "Age": Column(float, Check.greater_than(18)),
    "Gender": Column(str, Check.isin(["Male", "Female"])),
    "Education Level": Column(str),
    "Job Title": Column(str),
    "Years of Experience": Column(float, Check.greater_than_or_equal_to(0)),
    "Salary": Column(float, Check.greater_than(0)),
})

validated_df = schema.validate(df_clean)

# 3. Feature Engineering
df_clean['Exp_to_Age_Ratio'] = df_clean['Years of Experience'] / df_clean['Age']

# 4. Train/Test Split (80/20)
X = df_clean.drop(columns=['Salary'])
y = df_clean['Salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Cleaned training samples: {len(X_train)} | Test samples: {len(X_test)}")

Cleaned training samples: 259 | Test samples: 65


In [10]:
# ==========================================
# STEP 6: ML-READY DATA
# ==========================================
# Final dataset ready for machine learning algorithms
X = df_clean.drop(columns=['Salary'])
y = df_clean['Salary']

print("ML-Ready Features shape:", X.shape)
print("Target shape:", y.shape)

# ==========================================
# STEP 7: TRAIN / TEST SPLIT
# ==========================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

ML-Ready Features shape: (324, 6)
Target shape: (324,)
Training set: 259 samples
Testing set:  65 samples


In [11]:
import mlflow
import mlflow.sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 1. Define Column Preprocessor
num_features = ['Age', 'Years of Experience', 'Exp_to_Age_Ratio']
cat_features = ['Gender', 'Education Level', 'Job Title']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

# 2. Define Models to Experiment
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

# 3. MLflow Experiment Tracking
mlflow.set_experiment("Salary_Prediction_Experiment")

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):
        # Create full pipeline
        pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
        
        # Fit pipeline on training data
        pipeline.fit(X_train, y_train)
        
        # Evaluate model performance
        y_pred = pipeline.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        # Log parameters, metrics, and trained artifacts to MLflow
        mlflow.log_param("model_type", model_name)
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2_Score", r2)
        mlflow.sklearn.log_model(pipeline, artifact_path="model")
        
        print(f"{model_name:18s} | R2 Score: {r2:.4f} | RMSE: {rmse:.2f}")

2026/09/01 20:33:25 INFO mlflow.tracking.fluent: Experiment with name 'Salary_Prediction_Experiment' does not exist. Creating a new experiment.
2026/09/01 20:33:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 20:33:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Linear Regression  | R2 Score: 0.8698 | RMSE: 15687.61


2026/09/01 20:33:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 20:33:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Decision Tree      | R2 Score: 0.7853 | RMSE: 20143.71


2026/09/01 20:33:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 20:33:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random Forest      | R2 Score: 0.8794 | RMSE: 15097.77


2026/09/01 20:33:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 20:33:42 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Gradient Boosting  | R2 Score: 0.8913 | RMSE: 14333.62


In [12]:
import mlflow
from mlflow.tracking import MlflowClient

# Initialize MLflow Client
client = MlflowClient()

# Get experiment details
experiment = client.get_experiment_by_name("Salary_Prediction_Experiment")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id], 
    order_by=["metrics.R2_Score DESC"]
)

# Identify Best Model
best_run = runs[0]
best_run_id = best_run.info.run_id
best_model_name = best_run.data.params['model_type']
best_r2 = best_run.data.metrics['R2_Score']

print(f"🏆 Best Performing Model: {best_model_name}")
print(f"📊 Best R2 Score: {best_r2:.4f}")
print(f"🆔 Best Run ID: {best_run_id}")

# Register Best Model in MLflow Model Registry
model_uri = f"runs:/{best_run_id}/model"
registered_model = mlflow.register_model(model_uri=model_uri, name="SalaryPredictionModel")

Successfully registered model 'SalaryPredictionModel'.
2026/09/01 20:35:15 WARNING mlflow.tracking._model_registry.fluent: Run with id 8be76903d05c4c96b9568ea29eb0fc3c has no artifacts at artifact path 'model', registering model based on models:/m-a2a318d02c854a1793642550a5e6b47e instead


🏆 Best Performing Model: Gradient Boosting
📊 Best R2 Score: 0.8913
🆔 Best Run ID: 8be76903d05c4c96b9568ea29eb0fc3c


Created version '1' of model 'SalaryPredictionModel'.


In [13]:
# Load Registered Model for Predictions
loaded_model = mlflow.sklearn.load_model(model_uri)

# Create sample input
sample_employee = pd.DataFrame([{
    'Age': 29.0,
    'Gender': 'Female',
    'Education Level': "Master's",
    'Job Title': 'Data Analyst',
    'Years of Experience': 4.0,
    'Exp_to_Age_Ratio': 4.0 / 29.0
}])

# Predict Salary
predicted_salary = loaded_model.predict(sample_employee)
print(f"💰 Predicted Salary for sample employee: ${predicted_salary[0]:,.2f}")

💰 Predicted Salary for sample employee: $62,835.36


In [14]:
# 1. Initialize git repository
git init

# 2. Create a .gitignore file
cat <<EOT > .gitignore
.ipynb_checkpoints/
__pycache__/
*.pkl
.env
.venv/
salary_data_profiling.html
EOT

SyntaxError: invalid syntax (1479197497.py, line 2)